# ✅ Notebook 10 — End-to-End Pipeline Verification
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

This notebook verifies the entire pipeline works from scratch.

---

## 1. Verify All Data Files Exist

In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

files_to_check = [
    '../data/raw/pubmedqa_raw.csv',
    '../data/processed/pubmedqa_cleaned.csv',
    '../data/processed/pubmedqa_labelled.csv',
    '../data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss',
    '../data/embeddings/faiss_index/chunk_mapping.pkl',
    '../data/embeddings/faiss_index/chunk_mapping.csv',
    '../models/classifier/distilbert_classifier/config.json',
    '../reports/schema_validation_report.md',
    '../reports/eda_report.md',
    '../reports/classification_report.md',
    '../reports/evaluation_report.md',
    '../reports/model_development_doc.md',
]

print("📂 File Existence Check:")
print("─" * 60)
all_exist = True
for f in files_to_check:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    print(f"  {status} {f}")
    if not exists:
        all_exist = False

print(f"\n{'✅ All files present' if all_exist else '❌ Some files missing'}")

📂 File Existence Check:
────────────────────────────────────────────────────────────
  ✅ ../data/raw/pubmedqa_raw.csv
  ✅ ../data/processed/pubmedqa_cleaned.csv
  ✅ ../data/processed/pubmedqa_labelled.csv
  ✅ ../data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss
  ✅ ../data/embeddings/faiss_index/chunk_mapping.pkl
  ✅ ../data/embeddings/faiss_index/chunk_mapping.csv
  ✅ ../models/classifier/distilbert_classifier/config.json
  ✅ ../reports/schema_validation_report.md
  ✅ ../reports/eda_report.md
  ✅ ../reports/classification_report.md
  ✅ ../reports/evaluation_report.md
  ✅ ../reports/model_development_doc.md

✅ All files present


## 2. Verify Data Integrity

In [2]:
import pandas as pd

# Raw
df_raw = pd.read_csv('../data/raw/pubmedqa_raw.csv')
assert set(['instruction', 'input', 'output']).issubset(df_raw.columns)
print(f"✅ Raw data: {len(df_raw):,} rows, columns: {list(df_raw.columns)}")

# Cleaned
df_clean = pd.read_csv('../data/processed/pubmedqa_cleaned.csv')
assert list(df_clean.columns) == ['question', 'context', 'answer']
print(f"✅ Cleaned data: {len(df_clean):,} rows, columns: {list(df_clean.columns)}")

# Labelled
df_label = pd.read_csv('../data/processed/pubmedqa_labelled.csv')
assert list(df_label.columns) == ['question', 'context', 'answer', 'category']
assert df_label['category'].nunique() == 6
min_pct = (df_label['category'].value_counts() / len(df_label) * 100).min()
assert min_pct >= 1.0
print(f"✅ Labelled data: {len(df_label):,} rows, 6 categories, min {min_pct:.1f}%")

✅ Raw data: 10,000 rows, columns: ['instruction', 'input', 'output']
✅ Cleaned data: 10,000 rows, columns: ['question', 'context', 'answer']
✅ Labelled data: 10,000 rows, 6 categories, min 1.5%


## 3. Verify FAISS Index

In [3]:
import faiss
import pickle

index = faiss.read_index('../data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss')
print(f"✅ FAISS index: {index.ntotal:,} vectors, dimension {index.d}")

with open('../data/embeddings/faiss_index/chunk_mapping.pkl', 'rb') as f:
    mapping = pickle.load(f)
print(f"✅ Chunk mapping: {len(mapping):,} rows, columns: {list(mapping.columns)}")

assert index.ntotal == len(mapping), "FAISS vectors ≠ mapping rows"
print(f"✅ FAISS vectors match mapping rows")

✅ FAISS index: 10,000 vectors, dimension 384
✅ Chunk mapping: 10,000 rows, columns: ['chunk_id', 'question', 'context', 'answer', 'category', 'text_chunk']
✅ FAISS vectors match mapping rows


## 4. Verify Classifier

In [4]:
from src.classification.classifier import load_classifier

clf = load_classifier()

test_predictions = {
    "What are the symptoms of diabetes?": "Symptoms",
    "How is cancer diagnosed?": "Diagnosis",
    "What is the treatment for asthma?": "Treatment",
    "What are the side effects of ibuprofen?": "Medication",
    "How can obesity be prevented?": "Prevention",
    "What is the immune system?": "General",
}

print("\nClassifier predictions:")
correct = 0
for text, expected in test_predictions.items():
    predicted = clf.predict(text)
    match = "✅" if predicted == expected else "⚠️"
    if predicted == expected:
        correct += 1
    print(f"  {match} '{text[:45]}...' → {predicted} (expected: {expected})")

print(f"\nAccuracy on test cases: {correct}/{len(test_predictions)}")

Loading classifier from: D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\models\classifier\distilbert_classifier
✅ Classifier loaded | Classes: ['Diagnosis', 'General', 'Medication', 'Prevention', 'Symptoms', 'Treatment']

Classifier predictions:
  ✅ 'What are the symptoms of diabetes?...' → Symptoms (expected: Symptoms)
  ✅ 'How is cancer diagnosed?...' → Diagnosis (expected: Diagnosis)
  ✅ 'What is the treatment for asthma?...' → Treatment (expected: Treatment)
  ✅ 'What are the side effects of ibuprofen?...' → Medication (expected: Medication)
  ✅ 'How can obesity be prevented?...' → Prevention (expected: Prevention)
  ✅ 'What is the immune system?...' → General (expected: General)

Accuracy on test cases: 6/6


## 5. Verify RAG Pipeline

In [5]:
from src.rag.pipeline import build_rag_pipeline

rag = build_rag_pipeline()

result = rag.answer("What causes hypertension?")

assert len(result['answer']) > 0, "Empty answer"
assert result['disclaimer_present'] == True, "Missing disclaimer"
assert result['top_k'] == 5, f"Expected 5 sources, got {result['top_k']}"

print(f"✅ RAG pipeline works")
print(f"   Answer length: {len(result['answer_raw'])} chars")
print(f"   Sources: {result['top_k']}")
print(f"   Disclaimer: {result['disclaimer_present']}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading FAISS index: D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\data\embeddings\faiss_index\pubmedqa_index_flatl2.faiss
  Vectors in index: 10,000
Loading chunk mapping: D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\data\embeddings\faiss_index\chunk_mapping.pkl
Loading LLM: google/flan-t5-base


Token indices sequence length is longer than the specified maximum sequence length for this model (1004 > 512). Running this sequence through the model will result in indexing errors


✅ RAG Pipeline ready
✅ RAG pipeline works
   Answer length: 7 chars
   Sources: 5
   Disclaimer: True


## 6. Verify Integrated Pipeline (Classifier + RAG)

In [6]:
result = rag.answer_with_routing(
    "What are the symptoms of diabetes?",
    category=clf.predict("What are the symptoms of diabetes?")
)

assert result['category'] is not None
assert result['disclaimer_present'] == True
assert result['category_matched_sources'] >= 0

print(f"✅ Integrated pipeline works")
print(f"   Category: {result['category']}")
print(f"   Matched sources: {result['category_matched_sources']}/{result['top_k']}")
print(f"   Answer: {result['answer_raw'][:150]}...")

✅ Integrated pipeline works
   Category: Symptoms
   Matched sources: 5/5
   Answer: glycaemia, nephropathy and retinopathy...


## 7. Verify Evaluation Metrics Module

In [7]:
from src.evaluation.metrics import compute_bleu, compute_rouge, compute_improvement

bleu = compute_bleu(["the cat sat on the mat"], ["the cat is on the mat"])
rouge = compute_rouge(["the cat sat on the mat"], ["the cat is on the mat"])
improvement = compute_improvement(0.5, 0.7)

print(f"✅ Metrics module works")
print(f"   BLEU test: {bleu:.4f}")
print(f"   ROUGE-L test: {rouge:.4f}")
print(f"   Improvement test: {improvement:.1f}%")

✅ Metrics module works
   BLEU test: 0.2541
   ROUGE-L test: 0.8333
   Improvement test: 40.0%


## 8. Verify Reports Exist

In [8]:
reports = [
    ('../reports/schema_validation_report.md', 'Schema Validation'),
    ('../reports/eda_report.md', 'EDA Report'),
    ('../reports/classification_report.md', 'Classification Report'),
    ('../reports/evaluation_report.md', 'Evaluation Report'),
    ('../reports/model_development_doc.md', 'Model Development Doc'),
]

print("📄 Reports Check:")
for path, name in reports:
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"  ✅ {name}: {size:,} bytes")
    else:
        print(f"  ❌ {name}: MISSING")

📄 Reports Check:
  ✅ Schema Validation: 23,654 bytes
  ✅ EDA Report: 2,765 bytes
  ✅ Classification Report: 1,182 bytes
  ✅ Evaluation Report: 1,081 bytes
  ✅ Model Development Doc: 4,538 bytes


## 9. Full Pipeline — Single Query Demo

In [9]:
print("=" * 80)
print("🏥 FULL PIPELINE DEMO")
print("=" * 80)

demo_query = "Can bacterial gastroenteritis lead to irritable bowel syndrome?"

# Classify
category = clf.predict(demo_query)
print(f"\n📋 Query: {demo_query}")
print(f"🏷️  Category: {category}")

# RAG with routing
result = rag.answer_with_routing(demo_query, category=category)

print(f"\n📚 Retrieved {result['top_k']} sources ({result['category_matched_sources']} matched category)")
for s in result['retrieved_sources']:
    print(f"   Chunk {s['chunk_id']} | {s['category']} | dist: {s['distance']:.4f}")

print(f"\n💬 Answer:\n{result['answer']}")

🏥 FULL PIPELINE DEMO

📋 Query: Can bacterial gastroenteritis lead to irritable bowel syndrome?
🏷️  Category: General

📚 Retrieved 5 sources (0 matched category)
   Chunk 3 | Symptoms | dist: 0.5919
   Chunk 1198 | Symptoms | dist: 0.7844
   Chunk 6504 | Symptoms | dist: 0.8341
   Chunk 32 | Symptoms | dist: 0.9084
   Chunk 7982 | Symptoms | dist: 0.9289

💬 Answer:
IBS is a bacterial gastroenteritis. [Source 4]

⚠️ MEDICAL DISCLAIMER: This response is generated by an AI system for educational and informational purposes only. It is NOT a substitute for professional medical advice, diagnosis, or treatment. Always consult a qualified healthcare provider for medical decisions.


## ✅ End-to-End Verification Complete

| Component | Status |
|---|---|
| Data files | Checked |
| Data integrity | Verified |
| FAISS index | Loaded & matched |
| Classifier | Loaded & predicting |
| RAG pipeline | Generating answers |
| Integrated pipeline | Routing + retrieval working |
| Evaluation metrics | Computing correctly |
| Reports | All present |

**🎉 The full pipeline is working end-to-end!**

---

### Notebook Execution Order
```
01_data_loading.ipynb          → Load raw data
02_preprocessing.ipynb         → Clean & normalise
02b_category_labelling.ipynb   → Assign 6 medical categories
03_eda.ipynb                   → Exploratory data analysis
04_embeddings_vectorstore.ipynb → Build FAISS index
05_rag_pipeline.ipynb          → Build RAG pipeline
06_classification_model.ipynb  → Fine-tune DistilBERT
07_evaluation.ipynb            → Evaluate RAG vs baseline
08_integrated_pipeline.ipynb   → Wire classifier + RAG
10_end_to_end_test.ipynb       → This verification notebook
```